# 02 – Characters

Esplorazione e data cleaning del dataset `characters.csv`.

**Colonne:**

|---|---|
| Colonna | Descrizione |
| `character_mal_id` | ID univoco del personaggio su MyAnimeList |
| `url` | URL della pagina MAL del personaggio |
| `name` | Nome del personaggio |
| `name_kanji` | Nome in giapponese |
| `image` | URL dell'immagine del personaggio |
| `favorites` | Numero di utenti che hanno aggiunto il personaggio ai preferiti |
| `about` | Descrizione testuale del personaggio |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [2]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

df_characters = pd.read_csv('../datasets/characters.csv')
print(f'Shape: {df_characters.shape}')
print()
df_characters.info()
df_characters.head()

Shape: (209963, 7)

<class 'pandas.DataFrame'>
RangeIndex: 209963 entries, 0 to 209962
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   character_mal_id  209961 non-null  float64
 1   url               209961 non-null  str    
 2   name              209961 non-null  str    
 3   name_kanji        154483 non-null  str    
 4   image             209961 non-null  str    
 5   favorites         209961 non-null  float64
 6   about             112987 non-null  str    
dtypes: float64(2), str(5)
memory usage: 11.2 MB


,character_mal_id,url,name,name_kanji,image,favorites,about
0,280386.0,https://myanimelist.net/character/280386/Envi_...,Envi Mel Champagne,エンヴィ・メル・シャンパーニュ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
1,280354.0,https://myanimelist.net/character/280354/Eleven,Eleven,イレヴン,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
2,280353.0,https://myanimelist.net/character/280353/Stud,Stud,スタッド,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
3,280352.0,https://myanimelist.net/character/280352/Judge,Judge,ジャッジ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
4,280339.0,https://myanimelist.net/character/280339/Eiji_...,Eiji Kurokawa,黒川 英治,https://cdn.myanimelist.net/img/sp/icon/apple-...,0.0,NaN


Il dataset contiene **209.963** righe che corispondono ai personaggi e **7** colonne che corrispondo agli attributi per ciascuno.

Notiamo che il numero di righe non nulle non corrisponde al numero totale di righe, il che significa che ci sono dei valori mancanti da gestire.

Notiamo anche che `character_mal_id` e `favorite` sono di tipo `float64` invece di `int` il che potrebbe portare a errori, occupa più memoria del necessario e per numeri molto grandi potrebbe ridurre la leggibilità.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [3]:
n_originale = len(df_characters) #calcoliamo il numero totale di righe

mask_dup = df_characters.duplicated(keep=False)       # tutte le occorrenze coinvolte
n_righe_coinvolte = mask_dup.sum()                    # righe totali che hanno almeno un duplicato
n_gruppi = df_characters[mask_dup].duplicated(keep='first').sum()  # occorrenze extra (da rimuovere)
n_tenute = n_righe_coinvolte - n_gruppi               # prime occorrenze mantenute

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_characters.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_characters):,}')

Righe totali coinvolte in duplicazioni : 796
  → prime occorrenze mantenute         : 340
  → occorrenze extra rimosse           : 456

Righe prima della rimozione : 209,963
Righe dopo la rimozione     : 209,507


Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `character_mal_id`
 Questa è la chiave primaria del dataset: ogni riga deve avere un ID univoco e non nullo.

In [ ]:
analyze(df_characters['character_mal_id'])

**Osservazioni:**
- La colonna è `float64` invece di `int`, probabilmente dovuto alla presenza di valori null. Questo è un problema perché occupa più memoria del necessario e può ridurre la leggibilità per numeri grandi. Abbiamo verificato che non esistono valori di tipo `float`.
- Su 209.507 righe, c'è esattamente 1 valore null. Essendo la chiave primaria, non può essere tollerato e quindi quella riga va rimossa.
- Tutti i 209.506 valori non nulli sono unici (100% unicità), coerente con il ruolo di chiave primaria. I duplicati sono già stati eliminati in precedenza.
- Non ci sono outliers. Con IQR = 150.688, le soglie outlier sarebbero -151.636 e 451.116. I nostri ID vanno da 1 a 282.284. Entrambi rientrano dentro le soglie.

**Pulizie necessarie:**
- Convertiamo i valori di tipo `float64` a `int`.
- Rimuoviamo le righe con `character_mal_id` null in quanto è chiave primaria.
- Non ci sono duplicati da rimuovere in quanto sono già stati gestiti durante la
  rimozione dei duplicati esatti.

In [ ]:
df_characters.dropna(subset=['character_mal_id'], inplace=True)
non_interi = (df_characters['character_mal_id'] % 1 != 0).sum()
print(f'Valori non interi: {non_interi}')
df_characters['character_mal_id'] = df_characters['character_mal_id'].astype(int)
print(f'Righe dopo pulizia character_mal_id: {len(df_characters):,}')

### 2.2 `url`
Questa colonna contiene il link alla pagina del personaggio su MyAnimeList. Utilizziamo `strip()` per rimuovere eventuali spazzi vuoti all inizio e fine del URL. Effetuiamo poi l'analisi usando la nostra libreria `dataset_analyzer` e facciamo un controllo sul format del URL. Controlliamo anche la consistenza `character_mal_id` ↔ `url`. L'URL contiene l'ID del personaggio nel percorso (`/character/{id}/`). Verifichiamo che l'ID estratto dall'URL corrisponda al valore di `character_mal_id` per ogni riga.

In [ ]:
df_characters['url'] = df_characters['url'].str.strip()
analyze(df_characters['url'])

pattern = r'^https://myanimelist\.net/character/\d+/\S+$'
non_conformi = df_characters[~df_characters['url'].str.match(pattern)]
print(f'URL non conformi al formato atteso: {len(non_conformi)}')
if len(non_conformi) > 0:
    print(non_conformi[['character_mal_id', 'url', 'name']].to_string(index=True))

id_from_url = df_characters['url'].str.extract(r'/character/(\d+)/')[0].astype(int)
mismatch = (id_from_url != df_characters['character_mal_id'])
print()
print(f'Righe con ID non coerente tra url e character_mal_id: {mismatch.sum()}')

**Osservazioni**
- Non ci sono dei valori nulli o duplicati.
- Tutti i 209.506 URL sono unici (100%).
- Lunghezza varia tra 39 e 94 caratteri. La variazione è interamente dovuta alla lunghezza del nome in coda all'URL.
- Nessuna inconsistenza `character_id` - `URL` trovata: l'ID estratto dall'URL corrisponde esattamente a `character_mal_id` per tutte le 209.506 righe.

**Nessuna pulizia necessaria.**

### 2.3 `name`

Nome del personaggio. A differenza di `character_mal_id`, i duplicati sono attesi in quanto personaggi diversi possono condividere lo stesso nome.
Applichiamo `str.strip()` per rimuovere eventuali spazi o caratteri invisibili a inizio e fine stringa, come già fatto per `url`.

In [ ]:
df_characters['name'] = df_characters['name'].str.strip()
analyze(df_characters['name'])

**Osservazioni**
- Non ci sono stringhe null o vuote.
- Ci sono 46.719 duplicati (22.3%) che sono attesi e non problematici.
- Ci sono solo 162.787 nomi unici su 209.506 che conferma che i nomi non sono identificatori univoci.
- Lunghezza minima è di 1 carattere. Lunghezza massima è di 54 caratteri. Effetuiamo un controllo più approfondito.
- Nomi contenenti `@`: sono solo 13 quindi possiamo verificare se si tratta di nomi legittimi o dati corrotti.
- Nomi solo numerici: verifichiamo se si tratta di designazioni valide o placeholder.

In [ ]:
cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
lunghezze = df_characters['name'].dropna().str.len()

# Nomi più corti
idx_corti = lunghezze.nsmallest(10).index
print(f"10 nomi più corti:")
print(df_characters.loc[idx_corti].drop(columns=cols_to_drop).to_string())

#Nomi più lunghi
idx_lunghi = lunghezze.nlargest(10).index
print(f"\n10 nomi più lunghi:")
print(df_characters.loc[idx_lunghi].drop(columns=cols_to_drop).to_string())
print()

# Nomi contenenti '@'
nomi_con_at = df_characters[df_characters['name'].str.contains('@|＠', regex=True)]
print(f"Nomi contenenti '@': {len(nomi_con_at)}")
if len(nomi_con_at) > 0:
    cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
    print(nomi_con_at.drop(columns=cols_to_drop).to_string())

# Nomi solo numerici
nomi_numerici = df_characters[df_characters['name'].str.fullmatch(r'\d+')]
print(f'\nNomi solo numerici: {len(nomi_numerici)}')
if len(nomi_numerici) > 0:
    cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
    print(nomi_numerici.drop(columns=cols_to_drop).to_string())

- **Nomi di lunghezze estreme**: Tutti i valori estremi corrispondono ai nomi corretti.
- **Nomi contenenti `@`**: Dall'ispezione si tratta di un separatore usato per distinguere il nome del personaggio dal gruppo/ruolo di appartenenza (es. `Doris@Doll`, `Joe@Admin`, `Devo@Leukocyte`). L'unica eccezione è il personaggio `@Gou` ma anche in questo caso, dalle verifiche risulta che il nome è corretto. Non sono errori.
- **Nomi solo numerici**: sono presenti 26 nomi composti esclusivamente da cifre (es. `777`, `07`, `004989`). In contesti anime è comune che personaggi abbiano designazioni numeriche come nome ufficiale. Dall'ispezione sembrano essere nomi validi.

**Nessuna pulizia necessaria.**

### 2.4 `name_kanji`
Nome del personaggio in caratteri giapponesi. Applichiamo `str.strip()` per rimuovere eventuali spazi o caratteri invisibili a inizio e fine stringa.

In [ ]:
df_characters['name_kanji'] = df_characters['name_kanji'].str.strip()
analyze(df_characters['name_kanji'])

**Osservazioni**
- Ci sono 55.368 null (26.4%). Questo è un valore atteso che si può riferire a personaggi occidentali o di origine non giapponese che non hanno un nome kanji.
- Ci sono 24.972 duplicati che come per name, sono attesi.
- Lunghezza minima è di 1 carattere che come per `name`, è valido.
- Lunghezza massima è di 47 caratteri. Ci sono personaggi con alias multipli.
- Ci sono dei nomi contenenti il carattere `@` oppure solo caratteri numerici.Da verificare se si tratta di errori.

In [ ]:
# name_kanji contenenti '@'
nk_con_at = df_characters[df_characters['name_kanji'].notna() & df_characters['name_kanji'].str.contains('@|＠', regex=True)]
print(f"name_kanji contenenti '@': {len(nk_con_at)}")
if len(nk_con_at) > 0:
    cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
    print(nk_con_at.drop(columns=cols_to_drop).to_string())

# name_kanji solo numerici
nk_numerici = df_characters[df_characters['name_kanji'].notna() & df_characters['name_kanji'].str.fullmatch(r'\d+')]
print(f'\nname_kanji solo numerici: {len(nk_numerici)}')
if len(nk_numerici) > 0:
    cols_to_drop = [c for c in ['image', 'favorites', 'about'] if c in df_characters.columns]
    print(nk_numerici.drop(columns=cols_to_drop).to_string())

**Osservazioni**
- **`name_kanji` contenenti `@`**: Coerentemente con quanto osservato in `name`, il simbolo è usato come separatore nome/gruppo (es. `エルザ@ドール`, `軟体系@アキバ`). Non sono errori.
- **`name_kanji` solo numerici**: sono presenti 7 valori composti solo da cifre (es. `07`, `01`, `245`). Corrispondono ai medesimi personaggi già identificati in `name` con designazione numerica. Non sono errori.

**Nessuna pulizia necessaria.**

### 2.5 `image`
URL dell'immagine del personaggio su MyAnimeList. Effetuiamo l'analisi usando la nostra libreria `dataset_analyzer` dopo aver usato `strip()` per rimuovere spazi vuoti e facciamo poi un controlo sul format del URL.

In [ ]:
df_characters['image'] = df_characters['image'].str.strip()
analyze(df_characters['image'])

pattern_img = r'^https://cdn\.myanimelist\.net/(images/characters/\d+/\d+\.jpg|img/sp/icon/apple-touch-icon-256\.png)$'
non_conformi_img = df_characters[~df_characters['image'].str.match(pattern_img)]
print(f'Immagini non conformi al formato atteso: {len(non_conformi_img)}')
if len(non_conformi_img) > 0:
    print(non_conformi_img[['character_mal_id', 'image', 'name']].to_string(index=True))


**Osservazione**
- Non ci sono stringhe vuote o null. Tutta la colonna è popolata.
- Ci sono 23.179 URL duplicati (11.1%). Tutti corrispondono a un link che porta a un'immagine placeholder. A ogni personaggio senza immagine reale viene assegnato il logo MAL. Non è un errore e non è necessaria nessuna pulizia.
- Ci sono 186.327 URL unici (88.94%). Tutti i personaggi con immagine reale hanno un URL univoco.
- Tutte le URL seguono il formato corretto senza nessuna anomalia strutturale.
- Nella distribuzione delle lunghezze, le fasce 59–63 hanno 0 occorrenze, il che significa che le URL si dividono in due gruppi: percorso /images/characters/ (55–58 caratteri) e percorso placeholder /img/sp/icon/ (64 caratteri).


**Nessuna pulizia necessaria**

### 2.6 `favorites`

Numero di utenti che hanno aggiunto il personaggio ai propri preferiti. È una metrica di popolarità.

In [ ]:
analyze(df_characters['favorites'])

**Osservazioni**
- Nessun null, nessun valore negativo. La colonna è completamente popolata e non contiene errori.
- Il `dtype` è `float64` invece di `int64`, come già visto per `character_mal_id`.
- Il 62.85% dei valori è 0 (131.681 personaggi). Questo è un valore atteso. Significa che la grande maggioranza dei personaggi non è stata aggiunta nei preferiti come per esemio personaggi di serie nuove.
- Si nota una distribuzione asimetrica con Mediana = 0, Media = 57.7 e Coefficiente di Variazione = 2077%. Significa che pochi personaggi vengo aggiunti di più nei preferiti.
- P95 = 47 significa che il 95% dei personaggi ha ≤ 47 preferiti.
- Si notano 30,242 outlier secondo IQR. Non sono dati sporchi ma semplicemente il metodo IQR non è adatto a questa colonna. Qualsiasi valore maggiore di 5 viene classificato outlier ma avere più di 5 preferiti non è un anomalia.
- Ci sono 2.253 valori unici su 209.506 righe (1.08%). Non è un errore in quanto molti personaggi condividono gli stessi conteggi bassi.

**Pulizie necessarie:**
- Convertire il `dtype` float64 a `int`

In [ ]:
non_interi = (df_characters['favorites'] % 1 != 0).sum()
print(f'Valori non interi: {non_interi}')
df_characters['favorites'] = df_characters['favorites'].astype(int)
print(f'favorites dtype   : {df_characters["favorites"].dtype}')

Stampiamo un campione di righe con valori estremi (più grandi e più piccoli) per verificare se si osserva qualche anomalia.

In [ ]:
cols_to_drop = [c for c in ['image', 'about'] if c in df_characters.columns]

print("10 personaggi con più favorites:")
print(df_characters.nlargest(10, 'favorites').drop(columns=cols_to_drop).to_string())
print()
print("10 personaggi con meno favorites (escludendo 0):")
non_zero = df_characters[df_characters['favorites'] > 0]
print(non_zero.nsmallest(10, 'favorites').drop(columns=cols_to_drop).to_string())

I valori estremi non rappresentano errori in quanto corrispondono a personaggi molto famosi oppure nuovi/meno conosciuti.

### 2.7 `about`
Testo libero con la descrizione del personaggio.

In [ ]:
df_characters['about'] = df_characters['about'].str.strip()
analyze(df_characters['about'])

**Osservazioni**
- Notiamo che 46.19% è null (96.770). Questo risultato è atteso in quanto i personaggi secondari o poco conosciuti possono non avere una descrizione.
- Ci sono 3,221 duplicati che controlliamo più in dettaglio successivamente.
- Tra le stringhe più corte ci sono `"ANN"` e `"None"` che verifichiamo successivamente per determinare se vanno rimosse.
- Il 87.7% dei valori non-null ha lunghezza ≤ 725 caratteri. La maggior parte delle descrizioni è breve. Solo il 10.6% supera i 725 caratteri. Controlliamo i testi più lunghi per vedere se ci sono anomalie.
- Risulta 1 URL e 120 valori contenenti `@`. Questo può essere rumore minore, probabilmente link o menzioni. Verifichiamo più in dettaglio.
- Lingua prevalentemente inglese (parole più comuni: the, to, and, a, of…), ma i 2.314 caratteri unici indicano presenza di testo non-latino (giapponese, arabo, ecc.).

In [ ]:
# Valori duplicati
dup_mask = df_characters['about'].duplicated(keep=False) & df_characters['about'].notna()
dup_texts = df_characters.loc[dup_mask, 'about'].value_counts()
print(f'Testi duplicati distinti: {len(dup_texts)}')
print(dup_texts.head(20))

cols_to_drop = [c for c in ['image', 'favorites'] if c in df_characters.columns]

lunghezze = df_characters['about'].dropna().str.len()
print()

# Testi più corti
print("10 testi più corti:")
idx_corti = lunghezze.nsmallest(10).index
print(df_characters.loc[idx_corti].drop(columns=cols_to_drop).to_string())

# Testi più lunghi
print("\n10 testi più lunghi:")
idx_lunghi = lunghezze.nlargest(10).index
print(df_characters.loc[idx_lunghi].drop(columns=cols_to_drop).to_string())

cols_to_drop_about = [c for c in ['image', 'favorites'] if c in df_characters.columns]

# Testi contenenti URL
con_url = df_characters[df_characters['about'].str.contains(r'https?://', regex=True, na=False)]
print(f'Testi contenenti URL: {len(con_url)}')
if len(con_url) > 0:
    print(con_url.drop(columns=cols_to_drop_about).to_string())

# Testi contenenti '@'
con_at = df_characters[df_characters['about'].str.contains('@', regex=False, na=False)]
print(f"\nTesti contenenti '@': {len(con_at)}")
if len(con_at) > 0:
    print(con_at.head(10).drop(columns=cols_to_drop_about).to_string())

- Ci sono 3.221 duplicati. Da questi, la frase "No voice actors have been added to this character. Help improve our database…" compare 199 volte e rappresenta un placeholder che va rimosso. Abbiamo indagato più in dettaglio e sembra che il resto dei duplicati è composto dalla frase "Appears in episode X.". Questa frase, pur essendo ripetitiva, fornisce un'informazione utile (l'episodio in cui appare il personaggio) e non va rimossa.
-  La stringa più corta è `"ANN"` (3 caratteri): tag residuo senza significato descrittivo, da convertire in `null`. È presente anche `"None"`, anch'essa da convertire in `null`
- I testi contenenti @ e URL sono validi in quanto fanno riferimento a mail oppure fonti d'informazioni e dunque vengono tenuti.

**Pulizia necessaria:** rimozione dei testi boilerplate e dei valori placeholder (`"ANN"`, `"None."`).

In [ ]:
placeholder_patterns = [
    r'^No voice actors have been added to this character\.',
    r'^ANN$',
    r'^None\.$',
]
mask_placeholder = df_characters['about'].str.contains(
    '|'.join(placeholder_patterns), regex=True, na=False
)
print(f'Righe con testo placeholder: {mask_placeholder.sum()}')
df_characters.loc[mask_placeholder, 'about'] = np.nan
print(f'Null in about dopo pulizia: {df_characters["about"].isna().sum():,}')

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effetuiamo il salvataggio del dataset finale.

In [ ]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_characters):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_characters):>10,}')

print()
df_characters.to_csv('../datasets_cleaned/characters_clean.csv', index=False)
print('Salvato: datasets_cleaned/characters_clean.csv')

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effetuiamo il salvataggio del dataset finale.

In [ ]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_characters):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_characters):>10,}')

print()
df_characters.to_csv('../datasets_cleaned/characters_clean.csv', index=False)
print('Salvato: datasets_cleaned/characters_clean.csv')